# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import joblib

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

In [2]:
df = pd.read_csv("../data/day-of-week-not-scaled.csv")
df_days = pd.read_csv("../data/dayofweek.csv")
df["dayofweek"] = df_days["dayofweek"].values

X = df.drop(columns="dayofweek")
y = df["dayofweek"]

Первый шаг (Выделяем чистый тест): Вы берете исходные X и y и отрезаете от них 
данных. Получаются X_test и y_test (их мы откладываем в сторону до самого финала). Оставшиеся 
временно называются X_train и y_train.
Второй шаг (Выделяем валидацию): Вы берете полученные на первом шаге X_train и y_train (
от исходного датасета) и снова отправляете их в train_test_split. От них отрезается еще кусочек (обычно тоже), который перезаписывается в X_valid и y_valid, а остаток окончательно становится вашим рабочим X_train и y_train.


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train, test_size=0.2, random_state=21, stratify=y_train
)

## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [10]:
svc_best_params = {
    "C": 10,
    "class_weight": None,
    "gamma": "auto",
    "kernel": "rbf",
    "probability": True,
    "random_state": 21,
}

dt_best_params = {
    "class_weight": "balanced",
    "criterion": "gini",
    "max_depth": 21,
    "random_state": 21,
}

rf_best_params = {
    "class_weight": "balanced",
    "criterion": "entropy",
    "max_depth": 24,
    "n_estimators": 100,
    "random_state": 21,
}

In [11]:
svc_model = SVC(**svc_best_params).fit(X_train, y_train)
dt_model = DecisionTreeClassifier(**dt_best_params).fit(X_train, y_train)
rf_model = RandomForestClassifier(**rf_best_params).fit(X_train, y_train)

In [13]:
y_pred_svc = svc_model.predict(X_valid)
y_pred_dt = dt_model.predict(X_valid)
y_pred_rf = rf_model.predict(X_valid)

In [ ]:
for model_name, y_pred in zip(("svc", "dt", "rf"), (y_pred_svc, y_pred_dt, y_pred_rf)):
    acc = accuracy_score(y_valid, y_pred)
    precision = precision_score(y_valid, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_valid, y_pred, average="weighted")
    print(f"Метрики для {model_name}:")
    print(f"accuracy is {acc:.5f}")
    print(f"precision is {precision:.5f}")
    print(f"recall is {recall:.5f}\n")

Метрики для svc:
accuracy is 0.87778
precision is 0.88162
recall is 0.87778

Метрики для dt:
accuracy is 0.86667
precision is 0.87170
recall is 0.86667

Метрики для rf:
accuracy is 0.89630
precision is 0.89698
recall is 0.89630



## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.
2. Play with the other parameteres.
3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.
2. Play with the other parameters.
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.